# Classifieur de Validité des Phrases de Trajet avec CamemBERT

Ce notebook entraîne un classifieur binaire utilisant **transfer learning** avec **CamemBERT** pour détecter si une phrase est **valide** (contient une intention de trajet) ou **invalide** (hors contexte).

## Objectif
Créer un système qui filtre les phrases avant le traitement NLP, permettant de ne traiter que les phrases pertinentes.

## Avantages de CamemBERT
- **Compréhension contextuelle** : Meilleure compréhension du sens des phrases
- **Transfer learning** : Modèle pré-entraîné sur du français
- **Performance** : Généralement meilleure que les modèles classiques (TF-IDF + Logistic Regression)

## Structure
1. **Configuration & Imports** - Paramètres centralisés et dépendances
2. **Chargement des Données** - Lecture et exploration du dataset
3. **Preprocessing** - Tokenisation avec CamemBERT
4. **Entraînement avec Fine-tuning** - Fine-tuning de CamemBERT avec checkpoints
5. **Évaluation** - Métriques et analyse des erreurs
6. **Sauvegarde du Modèle Final** - Export pour production

## 1. Configuration & Imports

### Configuration centralisée

**Objectif** : Définir tous les paramètres en un seul endroit pour faciliter les expérimentations.

**Ce qu'on fait** :
- Définir les chemins vers les fichiers et dossiers (dataset, modèles, résultats, checkpoints)
- Configurer les paramètres du dataset (limite, seed pour reproductibilité)
- Définir comment diviser les données (train/validation/test)
- Configurer les paramètres de CamemBERT (max_length, batch_size, learning_rate, etc.)
- Activer/désactiver les checkpoints

**Pourquoi** : Avoir tous les paramètres au même endroit permet de modifier facilement l'expérience sans chercher dans tout le code.

In [ ]:
# ============================================================================
# CONFIGURATION - Modifier ces paramètres selon vos besoins
# ============================================================================

import os
from pathlib import Path
from datetime import datetime

# Chemins des fichiers
PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
DATASET_PATH = PROJECT_ROOT / "dataset" / "classifier" / "json" / "dataset.jsonl"
MODELS_DIR = PROJECT_ROOT / "classifier" / "models"
CHECKPOINTS_DIR = PROJECT_ROOT / "classifier" / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "classifier" / "results"
LOGS_DIR = PROJECT_ROOT / "classifier" / "logs" / "training"

# Créer les dossiers si nécessaire
for dir_path in [MODELS_DIR, CHECKPOINTS_DIR, RESULTS_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Paramètres du dataset
DATASET_LIMIT = None  # None pour utiliser tout le dataset, ou int pour limiter
RANDOM_SEED = 42

# Paramètres de split
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # Proportion du train qui devient validation
STRATIFY = True  # Maintenir la distribution des classes

# Paramètres CamemBERT
MODEL_NAME = "camembert-base"  # Modèle pré-entraîné à utiliser
MAX_LENGTH = 128  # Longueur maximale des séquences (tokens)
BATCH_SIZE = 16  # Taille des batches pour l'entraînement
LEARNING_RATE = 2e-5  # Taux d'apprentissage (standard pour fine-tuning)
NUM_EPOCHS = 3  # Nombre d'époques d'entraînement
WARMUP_STEPS = 100  # Nombre de steps pour le warmup
WEIGHT_DECAY = 0.01  # Régularisation L2

# Paramètres d'entraînement
SAVE_CHECKPOINTS = True  # Activer les checkpoints
CHECKPOINT_INTERVAL = 0.25  # Sauvegarder tous les 25% de l'entraînement
EARLY_STOPPING = True  # Early stopping
EARLY_STOPPING_PATIENCE = 2  # Nombre d'époques sans amélioration avant d'arrêter
SAVE_FINAL_MODEL = True  # Sauvegarder automatiquement le modèle final

# Métadonnées
EXPERIMENT_NAME = f"validity_classifier_camembert_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
VERSION = "v1"

print(f"✅ Configuration chargée")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Modèle: {MODEL_NAME}")
print(f"   Expérience: {EXPERIMENT_NAME}")

### Imports

**Objectif** : Charger toutes les bibliothèques nécessaires pour le transfer learning avec transformers.

**Ce qu'on fait** :
- **transformers** : Bibliothèque Hugging Face pour les modèles pré-entraînés (CamemBERT)
- **torch** : PyTorch pour l'entraînement
- **pandas/numpy** : Manipulation des données
- **scikit-learn** : Métriques d'évaluation
- **tqdm** : Barres de progression

**Pourquoi** : Ces bibliothèques fournissent toutes les fonctions nécessaires pour charger CamemBERT, fine-tuner le modèle, évaluer les performances et sauvegarder les résultats.

In [ ]:
# Standard libraries
import json
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Transformers (Hugging Face)
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split

# Utilitaires
from tqdm import tqdm

# Vérifier la disponibilité du GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports terminés")
print(f"   Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2. Chargement des Données

**Objectif** : Lire le dataset JSONL et le convertir en DataFrame pandas.

**Ce qu'on fait** :
- Créer une fonction `load_jsonl()` qui lit le fichier ligne par ligne
- Chaque ligne du JSONL contient un objet JSON avec : `sentence`, `departure`, `arrival`, `is_valid`
- Convertir toutes les lignes en un DataFrame pandas
- Afficher un aperçu des premières lignes pour vérifier que tout est correct

**Pourquoi** : Le format JSONL (JSON Lines) est pratique pour stocker de grandes quantités de données. Le DataFrame pandas permet ensuite de manipuler facilement les données.

In [ ]:
# Chargement du dataset JSONL
def load_jsonl(file_path, limit=None):
    """Charge un fichier JSONL et retourne un DataFrame."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)

# Charger les données
print(f"📂 Chargement du dataset depuis {DATASET_PATH}...")
df = load_jsonl(DATASET_PATH, limit=DATASET_LIMIT)
print(f"✅ {len(df)} phrases chargées")

# Afficher les premières lignes
print("\n📊 Aperçu des données:")
print(df.head())

### Exploration des données

**Objectif** : Comprendre les caractéristiques du dataset avant de commencer l'entraînement.

**Ce qu'on fait** :
1. **Statistiques de base** : Compter combien de phrases sont valides vs invalides
2. **Analyse de longueur** : Mesurer la longueur des phrases
3. **Exemples** : Afficher quelques exemples de phrases valides et invalides

In [ ]:
# Statistiques de base
print("📊 Statistiques du dataset:")
print(f"   Total de phrases: {len(df)}")
print(f"   Phrases valides: {df['is_valid'].sum()} ({(df['is_valid'].sum()/len(df)*100):.1f}%)")
print(f"   Phrases invalides: {(~df['is_valid'].astype(bool)).sum()} ({(~df['is_valid'].astype(bool)).sum()/len(df)*100):.1f}%)")

# Analyse de longueur
df['length'] = df['sentence'].str.len()
print(f"\n📏 Longueur des phrases:")
print(f"   Moyenne: {df['length'].mean():.1f} caractères")
print(f"   Médiane: {df['length'].median():.1f} caractères")
print(f"   Min: {df['length'].min()} caractères")
print(f"   Max: {df['length'].max()} caractères")

# Exemples
print("\n✅ Exemples de phrases VALIDES:")
for i, row in df[df['is_valid'] == 1].head(3).iterrows():
    print(f"   - {row['sentence']}")

print("\n❌ Exemples de phrases INVALIDES:")
for i, row in df[df['is_valid'] == 0].head(3).iterrows():
    print(f"   - {row['sentence']}")

## 3. Preprocessing avec CamemBERT

**Objectif** : Préparer les données pour l'entraînement avec CamemBERT.

**Ce qu'on fait** :
1. **Split train/validation/test** : Diviser les données
2. **Charger le tokenizer CamemBERT** : Tokeniser les phrases
3. **Créer un Dataset PyTorch** : Classe personnalisée pour charger les données efficacement

**Pourquoi** : CamemBERT nécessite une tokenisation spécifique et des formats de données particuliers pour fonctionner efficacement.

In [ ]:
# Split train/test
X = df['sentence'].values
y = df['is_valid'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y if STRATIFY else None
)

# Split train/validation
val_size_adjusted = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_size_adjusted,
    random_state=RANDOM_SEED,
    stratify=y_temp if STRATIFY else None
)

print(f"📦 Split des données:")
print(f"   Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"   Validation: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"   Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\n   Distribution train: {np.bincount(y_train)}")
print(f"   Distribution validation: {np.bincount(y_val)}")
print(f"   Distribution test: {np.bincount(y_test)}")

In [ ]:
# Charger le tokenizer CamemBERT
print(f"🔄 Chargement du tokenizer {MODEL_NAME}...")
tokenizer = CamembertTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer chargé")
print(f"   Vocabulaire: {len(tokenizer)} tokens")

# Test de tokenisation
test_sentence = "Je vais de Paris à Lyon"
encoded = tokenizer(test_sentence, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt')
print(f"\n🧪 Test de tokenisation:")
print(f"   Phrase: '{test_sentence}'")
print(f"   Tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0][:20])}...")
print(f"   Longueur: {encoded['input_ids'].shape[1]} tokens")

In [ ]:
# Classe Dataset personnalisée pour PyTorch
class ValidityDataset(Dataset):
    """Dataset pour la classification de validité des phrases."""
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # Tokeniser
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Créer les datasets
train_dataset = ValidityDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = ValidityDataset(X_val, y_val, tokenizer, MAX_LENGTH)
test_dataset = ValidityDataset(X_test, y_test, tokenizer, MAX_LENGTH)

print(f"✅ Datasets créés:")
print(f"   Train: {len(train_dataset)} exemples")
print(f"   Validation: {len(val_dataset)} exemples")
print(f"   Test: {len(test_dataset)} exemples")

## 4. Chargement et Configuration du Modèle

**Objectif** : Charger le modèle CamemBERT pré-entraîné et le configurer pour la classification binaire.

**Ce qu'on fait** :
1. Charger `CamembertForSequenceClassification` avec 2 classes (valide/invalide)
2. Configurer le modèle pour utiliser le GPU si disponible
3. Afficher le nombre de paramètres du modèle

**Pourquoi** : CamemBERT est pré-entraîné sur du texte français, on va juste fine-tuner la dernière couche pour notre tâche de classification.

In [ ]:
# Charger le modèle CamemBERT
print(f"🔄 Chargement du modèle {MODEL_NAME}...")
model = CamembertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # Classification binaire (valide/invalide)
    problem_type="single_label_classification"
)

# Déplacer le modèle sur le device (GPU ou CPU)
model.to(device)

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Modèle chargé")
print(f"   Device: {device}")
print(f"   Paramètres totaux: {total_params:,}")
print(f"   Paramètres entraînables: {trainable_params:,}")

## 5. Configuration de l'Entraînement

**Objectif** : Configurer les paramètres d'entraînement avec Hugging Face Trainer.

**Ce qu'on fait** :
1. Définir les métriques de calcul (accuracy, F1, precision, recall)
2. Configurer `TrainingArguments` (learning rate, batch size, epochs, etc.)
3. Configurer le `Trainer` avec callbacks (early stopping, checkpoints)

**Pourquoi** : Le Trainer de Hugging Face simplifie l'entraînement et gère automatiquement beaucoup de détails (optimisation, logging, etc.).

In [ ]:
# Fonction pour calculer les métriques
def compute_metrics(eval_pred):
    """Calcule les métriques d'évaluation."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='binary', zero_division=0)
    recall = recall_score(labels, predictions, average='binary', zero_division=0)
    f1 = f1_score(labels, predictions, average='binary', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("✅ Fonction de calcul des métriques définie")

In [ ]:
# Configuration des arguments d'entraînement
training_args = TrainingArguments(
    output_dir=str(CHECKPOINTS_DIR / EXPERIMENT_NAME),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    logging_dir=str(LOGS_DIR / EXPERIMENT_NAME),
    logging_steps=50,
    eval_strategy="epoch",  # Évaluer à la fin de chaque époque
    save_strategy="epoch",  # Sauvegarder à la fin de chaque époque
    save_total_limit=3,  # Garder seulement les 3 derniers checkpoints
    load_best_model_at_end=True,  # Charger le meilleur modèle à la fin
    metric_for_best_model="f1",  # Utiliser F1 comme métrique principale
    greater_is_better=True,
    report_to="none",  # Désactiver les rapports automatiques (wandb, etc.)
    seed=RANDOM_SEED,
    fp16=torch.cuda.is_available(),  # Utiliser la précision mixte si GPU disponible
)

print("✅ Arguments d'entraînement configurés")
print(f"   Output dir: {training_args.output_dir}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")

In [ ]:
# Créer le Trainer
callbacks = []
if EARLY_STOPPING:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=callbacks
)

print("✅ Trainer créé")
print(f"   Early stopping: {'Activé' if EARLY_STOPPING else 'Désactivé'}")

## 6. Entraînement du Modèle

**Objectif** : Fine-tuner CamemBERT sur notre dataset.

**Ce qu'on fait** :
1. Lancer l'entraînement avec `trainer.train()`
2. Le Trainer gère automatiquement :
   - L'optimisation (AdamW)
   - Le calcul des métriques
   - La sauvegarde des checkpoints
   - L'early stopping

**Pourquoi** : Le fine-tuning permet d'adapter le modèle pré-entraîné à notre tâche spécifique avec relativement peu de données.

In [ ]:
# Entraînement
print("🚀 Démarrage de l'entraînement...")
print("=" * 80)

train_result = trainer.train()

print("=" * 80)
print("✅ Entraînement terminé!")
print(f"   Loss finale: {train_result.training_loss:.4f}")
print(f"   Temps total: {train_result.metrics.get('train_runtime', 0):.1f}s")

## 7. Évaluation sur le Test Set

**Objectif** : Évaluer les performances du modèle sur le test set (données jamais vues).

**Ce qu'on fait** :
1. Faire des prédictions sur le test set
2. Calculer les métriques (accuracy, precision, recall, F1, ROC-AUC)
3. Afficher la matrice de confusion
4. Afficher le rapport de classification

**Pourquoi** : Le test set permet d'évaluer la capacité du modèle à généraliser sur de nouvelles données.

In [ ]:
# Évaluation sur le test set
print("📊 Évaluation sur le test set...")
print("=" * 80)

test_predictions = trainer.predict(test_dataset)
test_pred_labels = np.argmax(test_predictions.predictions, axis=1)
test_true_labels = test_predictions.label_ids

# Calculer les métriques
test_accuracy = accuracy_score(test_true_labels, test_pred_labels)
test_precision = precision_score(test_true_labels, test_pred_labels, average='binary', zero_division=0)
test_recall = recall_score(test_true_labels, test_pred_labels, average='binary', zero_division=0)
test_f1 = f1_score(test_true_labels, test_pred_labels, average='binary', zero_division=0)

# Probabilités pour ROC-AUC
test_probs = torch.softmax(torch.tensor(test_predictions.predictions), dim=1)[:, 1].numpy()
test_auc = roc_auc_score(test_true_labels, test_probs)

print(f"\n📈 Métriques sur le test set:")
print(f"   Accuracy:  {test_accuracy:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1-Score:  {test_f1:.4f}")
print(f"   ROC-AUC:   {test_auc:.4f}")

# Matrice de confusion
cm = confusion_matrix(test_true_labels, test_pred_labels)
print(f"\n📊 Matrice de confusion:")
print(f"   [[TN={cm[0,0]}, FP={cm[0,1]}],")
print(f"    [FN={cm[1,0]}, TP={cm[1,1]}]]")

# Rapport de classification
print(f"\n📋 Rapport de classification:")
print(classification_report(test_true_labels, test_pred_labels, target_names=['Invalide', 'Valide']))

## 8. Test sur des Phrases Exemples

**Objectif** : Tester le modèle sur des phrases spécifiques pour vérifier son comportement.

**Ce qu'on fait** :
1. Définir une liste de phrases de test (valides et invalides)
2. Faire des prédictions avec le modèle
3. Afficher les résultats avec les probabilités de confiance

**Pourquoi** : Cela permet de vérifier visuellement que le modèle fonctionne correctement sur des cas concrets.

In [ ]:
# Phrases de test (même liste que dans le notebook classique)
test_sentences = [
    # Phrases valides
    "Je vais de Paris à Lyon",
    "billet Paris Bordeaux svp",
    "train Marseille Nice demain",
    "Paris -> Lyon",
    "Je veux me rendre à Toulouse et partir depuis Bordeaux",
    "Trajet Lille - Marseille",
    "Donne moi le trajet SNCF suivant : Nantes - Strasbourg",
    "Je dois aller de Lyon à Paris demain",
    "billet Nice Cannes",
    "Quel est le trajet entre Rennes et Brest",
    "Je pars de Marseille pour aller à Paris",
    "Aller de Bordeaux à Toulouse svp",
    # Phrases invalides
    "Je mange une pomme",
    "Il fait beau aujourd'hui",
    "Quel temps fait-il demain",
    "Je vais au cinéma ce soir",
    "Mon ami m'a dit qu'il partait de Paris pour aller à Lyon",
    "Les trains sont en retard",
    "billet",
    "Je vais de la Terre à Mars",
    "Bonjour, comment allez-vous",
    "Il y a des travaux sur la voie",
]

expected_labels = {
    "Je vais de Paris à Lyon": 1,
    "billet Paris Bordeaux svp": 1,
    "train Marseille Nice demain": 1,
    "Paris -> Lyon": 1,
    "Je veux me rendre à Toulouse et partir depuis Bordeaux": 1,
    "Trajet Lille - Marseille": 1,
    "Donne moi le trajet SNCF suivant : Nantes - Strasbourg": 1,
    "Je dois aller de Lyon à Paris demain": 1,
    "billet Nice Cannes": 1,
    "Quel est le trajet entre Rennes et Brest": 1,
    "Je pars de Marseille pour aller à Paris": 1,
    "Aller de Bordeaux à Toulouse svp": 1,
    "Je mange une pomme": 0,
    "Il fait beau aujourd'hui": 0,
    "Quel temps fait-il demain": 0,
    "Je vais au cinéma ce soir": 0,
    "Mon ami m'a dit qu'il partait de Paris pour aller à Lyon": 0,
    "Les trains sont en retard": 0,
    "billet": 0,
    "Je vais de la Terre à Mars": 0,
    "Bonjour, comment allez-vous": 0,
    "Il y a des travaux sur la voie": 0,
}

print("🧪 Test du modèle sur des phrases exemples:")
print("=" * 80)

model.eval()
correct = 0
total = len(test_sentences)

for sentence in test_sentences:
    # Tokeniser
    encoded = tokenizer(
        sentence,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    # Prédire
    with torch.no_grad():
        outputs = model(**encoded)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        
    prediction = torch.argmax(probs, dim=1).item()
    confidence = probs[0][prediction].item()
    
    expected = expected_labels.get(sentence, -1)
    is_correct = prediction == expected
    if is_correct:
        correct += 1
    
    status = "✅" if is_correct else "❌"
    label = "VALIDE" if prediction == 1 else "INVALIDE"
    
    print(f"{status} {label} (confiance: {confidence:.2%}): {sentence}")

print("=" * 80)
print(f"\n📊 Résultats: {correct}/{total} correctes ({correct/total*100:.1f}%)")

## 9. Sauvegarde du Modèle Final

**Objectif** : Sauvegarder le modèle fine-tuné pour pouvoir l'utiliser en production.

**Ce qu'on fait** :
1. Sauvegarder le modèle et le tokenizer dans le dossier `models/`
2. Sauvegarder les métriques dans un fichier JSON
3. Créer un fichier de métadonnées avec les informations de l'expérience

**Pourquoi** : Il est important de sauvegarder le modèle pour pouvoir le réutiliser sans avoir à le réentraîner.

In [ ]:
if SAVE_FINAL_MODEL:
    print("💾 Sauvegarde du modèle final...")
    
    # Chemins de sauvegarde
    model_dir = MODELS_DIR / f"{EXPERIMENT_NAME}_{VERSION}"
    model_dir.mkdir(parents=True, exist_ok=True)
    
    # Sauvegarder le modèle et le tokenizer
    model.save_pretrained(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
    
    # Sauvegarder les métriques
    metrics = {
        'test_accuracy': float(test_accuracy),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall),
        'test_f1': float(test_f1),
        'test_auc': float(test_auc),
        'model_name': MODEL_NAME,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'experiment_name': EXPERIMENT_NAME,
        'version': VERSION
    }
    
    metrics_file = model_dir / "metrics.json"
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Modèle sauvegardé dans: {model_dir}")
    print(f"   Modèle: {model_dir / 'pytorch_model.bin'}")
    print(f"   Tokenizer: {model_dir / 'tokenizer.json'}")
    print(f"   Métriques: {metrics_file}")
else:
    print("ℹ️  Sauvegarde du modèle désactivée (SAVE_FINAL_MODEL=False)")

## 10. Chargement et Test du Modèle Sauvegardé

**Objectif** : Vérifier que le modèle sauvegardé peut être rechargé et fonctionne correctement.

**Ce qu'on fait** :
1. Recharger le modèle et le tokenizer depuis le fichier
2. Tester sur quelques phrases pour vérifier que tout fonctionne

**Pourquoi** : C'est important de vérifier que la sauvegarde/chargement fonctionne correctement avant de déployer en production.

In [ ]:
if SAVE_FINAL_MODEL:
    print("📂 Rechargement du modèle depuis le fichier...")
    
    # Recharger
    loaded_model = CamembertForSequenceClassification.from_pretrained(str(model_dir))
    loaded_tokenizer = CamembertTokenizer.from_pretrained(str(model_dir))
    loaded_model.to(device)
    loaded_model.eval()
    
    print("✅ Modèle rechargé")
    
    # Test rapide
    test_phrase = "Je vais de Paris à Lyon"
    encoded = loaded_tokenizer(
        test_phrase,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        outputs = loaded_model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()
    
    label = "VALIDE" if prediction == 1 else "INVALIDE"
    print(f"\n🧪 Test: '{test_phrase}' -> {label} (confiance: {confidence:.2%})")
    print("✅ Modèle fonctionne correctement!")
else:
    print("ℹ️  Modèle non sauvegardé, test de chargement ignoré")